# Fast PyMMM pipeline

Batched registration and extraction for mother-machine ND2 data. Edit the configuration cell, then run downward. `REGISTRATION_BACKEND="opencv"` uses fast iterative phase correlation; use `"stackreg"` for the original estimator. `REGISTRATION_ROI` is specified in unrotated raw-image coordinates and is intersected with any experiment ROI. Lane and trench detection remain interactive when no checkpoint exists.

In [1]:
from pathlib import Path
import json

from dask.distributed import Client, LocalCluster
from pymmm import (
    CompanionStore, Extractor, LaneDetector, ND2Experiment,
    Registrator, TrenchDetector,
)
from pymmm.registrator import TRANSFORM_CONVENTION

# Paths and optional input subsetting
ND2_PATH = Path("/data/20260331_SB5_6_7_8_16/20260331_restart.nd2")
OUTPUT_PATH = Path("/data/20260331_restart.zarr")   # the experiment group: one OME-Zarr image per trench
FOVS = None                 # e.g. [0, 1] or ["XYPos:0", "XYPos:1"]
TIME_RANGE = None           # e.g. (0, 20), end-exclusive
EXPERIMENT_Y_ROI = None     # e.g. (400, 1900)
EXPERIMENT_X_ROI = None     # e.g. (0, 2304)

# Parallelism and registration
WORKERS = 16                 # Dask worker processes; use 16 to match the parallel benchmarks
REGISTRATION_CHANNEL = "PC"
REGISTRATION_BACKEND = "stackreg" # "opencv" (fast) or "stackreg" (original)
REGISTRATION_MODE = "first"
REGISTRATION_JOBS = -1       # 1 = one ND2 reader; -1 = use the Dask WORKERS above
OPENCV_THREADS = 1          # OpenCV calls sharing each batch when REGISTRATION_JOBS=1
ROTATION = 0.0

# Coordinates are in the unrotated raw ND2 image; this is intersected with EXPERIMENT_*_ROI.
REGISTRATION_ROI = None     # stable high-contrast ROI, e.g. {"y": (600, 1650), "x": (650, 1650)}
REGISTRATION_DOWNSAMPLE = 4 # 4 is the measured fast default; use 2 for a conservative setting
REGISTRATION_BATCH_SIZE = 4
REGISTRATION_FOV_WINDOW = 2  # Dask path: overlap this many FOV task groups
REGISTRATION_MEAN_N_FRAMES = 10
REGISTRATION_MEAN_FROM = "end"
RECOMPUTE_REGISTRATION = False

# Detection and extraction
DETECTION_CHANNEL = "PC"
REDETECT_LANES = False
REDETECT_TRENCHES = False
EXTRACTION_BATCH_SIZE = 4
OUTPUT_TIME_CHUNK = None    # None keeps each full trench trajectory in one chunk
COMPRESSION_LEVEL = 3
BYTE_SHUFFLE = True
RESUME_EXTRACTION = True

In [2]:
if "client" in globals():
    client.close()
if "cluster" in globals():
    cluster.close()
cluster = LocalCluster(
    n_workers=WORKERS, threads_per_worker=1, processes=True,
)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 16
Total threads: 16,Total memory: 61.91 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:33167,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:36565,Total threads: 1
Dashboard: http://127.0.0.1:35257/status,Memory: 3.87 GiB
Nanny: tcp://127.0.0.1:38217,


2026-07-14 14:37:37,693 - tornado.application - ERROR - Uncaught exception GET /json/identity.json (127.0.0.1)
HTTPServerRequest(protocol='http', host='127.0.0.1:8787', method='GET', uri='/json/identity.json', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/home/georgeos/GitHub/PyMMM_2/.pixi/envs/default/lib/python3.14/site-packages/tornado/web.py", line 1857, in _execute
    result = method(*self.path_args, **self.path_kwargs)
  File "/home/georgeos/GitHub/PyMMM_2/.pixi/envs/default/lib/python3.14/site-packages/distributed/http/scheduler/json.py", line 58, in get
    self.write(self.server.identity())
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/georgeos/GitHub/PyMMM_2/.pixi/envs/default/lib/python3.14/site-packages/tornado/web.py", line 984, in write
    chunk = escape.json_encode(chunk)
  File "/home/georgeos/GitHub/PyMMM_2/.pixi/envs/default/lib/python3.14/site-packages/tornado/escape.py", line 96, in json_encode
    return json.dumps(

In [3]:
exp = ND2Experiment(ND2_PATH)
if FOVS is not None:
    exp.select_fovs(FOVS)
if TIME_RANGE is not None:
    exp.select_times(*TIME_RANGE)
if EXPERIMENT_Y_ROI is not None or EXPERIMENT_X_ROI is not None:
    exp.select_roi(y=EXPERIMENT_Y_ROI, x=EXPERIMENT_X_ROI)
store = CompanionStore.for_experiment(exp)
print(exp)
print(store)

ND2Experiment: 20260331_restart
  Path: /data/20260331_SB5_6_7_8_16/20260331_restart.nd2
  Dims: T=97 × P=75 × C=4 × Y=2304 × X=2304
  FOVs: 75  (XYPos:0, XYPos:1, XYPos:2...)
  Channels: PC, CFP, mVenus, mCherry
  Timepoints: 97
  Pixel size: 0.1079 µm
  Time interval: 300002.9 ms
CompanionStore(/data/20260331_SB5_6_7_8_16/20260331_restart.pymmm.zarr, [registration, lane_detection, trench_detection])


In [4]:
import ndv

class _LocalXarrayWrapper(ndv.DataWrapper):
    @classmethod
    def supports(cls, obj):
        return False

    @property
    def dims(self):
        return tuple(self._data.dims)

    @property
    def coords(self):
        return {dim: range(self._data.sizes[dim]) for dim in self.dims}

    def _asarray(self, data):
        array = data.data
        return array.compute(scheduler="threads") if hasattr(array, "compute") else array

    def channel_names(self, channel_axis):
        if channel_axis is None:
            return {}
        dim = self.dims[channel_axis]
        if dim not in self._data.coords:
            return {}
        return {i: str(value) for i, value in enumerate(self._data.coords[dim].values)}

raw_ndv_viewer = ndv.imshow(_LocalXarrayWrapper(exp.data))
raw_ndv_viewer

RFBOutputContext()

Unable to find extension: VK_EXT_physical_device_drm
No windowing system present. Using surfaceless platform
No config found!
No config found!
Max vertex attribute stride unknown. Assuming it is 2048


<IPython.core.display.Javascript object>

Max vertex attribute stride unknown. Assuming it is 2048


In [5]:
def _json_value(value):
    return json.loads(json.dumps(value, default=str))

desired_channel = (
    exp.channel_names[REGISTRATION_CHANNEL]
    if isinstance(REGISTRATION_CHANNEL, int) else REGISTRATION_CHANNEL
)
desired_registration = {
    "channel": desired_channel,
    "backend": REGISTRATION_BACKEND,
    "mode": REGISTRATION_MODE,
    "rotation": ROTATION,
    "roi": _json_value(REGISTRATION_ROI or {}),
    "mean_n_frames": REGISTRATION_MEAN_N_FRAMES,
    "mean_from": REGISTRATION_MEAN_FROM,
    "downsample": REGISTRATION_DOWNSAMPLE,
    "transform_convention": TRANSFORM_CONVENTION,
    "fov_names": exp.fov_names,
    "source_subset_metadata": exp.source_subset_metadata,
}
registration_compatible = False
if store.has_registration():
    saved = store.load_registration_params()
    if saved.get("backend") is None:
        saved["backend"] = "stackreg"
    if saved.get("transform_convention") is None and saved.get("rotation", 0) == 0:
        saved["transform_convention"] = TRANSFORM_CONVENTION
    registration_compatible = all(
        saved.get(key) == value for key, value in desired_registration.items()
    )

reset_registration = RECOMPUTE_REGISTRATION or not registration_compatible
if reset_registration:
    for section in ("registration", "lane_detection", "trench_detection"):
        if section in store._store:
            store.reset(section)

if store.has_registration():
    reg = Registrator.load(exp, store)
    print("Loaded compatible registration checkpoint")
else:
    reg = Registrator(
        exp, store, registration_channel=REGISTRATION_CHANNEL,
        mode=REGISTRATION_MODE, rotation=ROTATION, roi=REGISTRATION_ROI,
        mean_n_frames=REGISTRATION_MEAN_N_FRAMES,
        mean_from=REGISTRATION_MEAN_FROM, backend=REGISTRATION_BACKEND,
        downsample=REGISTRATION_DOWNSAMPLE,
        batch_size=REGISTRATION_BATCH_SIZE, fov_window=REGISTRATION_FOV_WINDOW,
        opencv_threads=OPENCV_THREADS,
    )
    reg.compute_tmats(plot=True, n_jobs=REGISTRATION_JOBS, )
    reg.save()
reg

Loaded compatible registration checkpoint


Registrator(channel='PC', backend='stackreg', mode='first', rotation=0.0, downsample=4, status=computed)

In [6]:
import xarray as xr

registered_data = xr.concat(
    [
        reg.get_stabilized_data(channel=channel).expand_dims(C=[channel])
        for channel in exp.channel_names
    ],
    dim="C",
)
registered_dims = [dim for dim in exp.data.dims if dim in registered_data.dims]
registered_data = registered_data.transpose(*registered_dims)
registered_ndv_viewer = ndv.imshow(_LocalXarrayWrapper(registered_data))
registered_ndv_viewer

RFBOutputContext()

<IPython.core.display.Javascript object>

## Lane detection

Browse/tune the candidate lanes in the widget. Click **Apply to all FOVs** before saving if you want to replace the current detections.

In [7]:
if REDETECT_LANES:
    for section in ("lane_detection", "trench_detection"):
        if section in store._store:
            store.reset(section)
if store.has_lane_detection():
    lane_det = LaneDetector.load(exp, reg, store)
    print(lane_det)
else:
    lane_det = LaneDetector(exp, reg, store, detection_channel=DETECTION_CHANNEL)

LaneDetector(150 lanes across 75 FOVs)


In [8]:
lane_det.interactive_detect_lanes()

RFBOutputContext()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
if not lane_det.is_detected:
    raise RuntimeError("Click 'Apply to all FOVs' in the lane widget first.")
lane_det.save()

Lane detection saved to /data/20260331_SB5_6_7_8_16/20260331_restart.pymmm.zarr


## Trench detection

Browse/tune the candidate trenches in the widget. Click **Apply to all FOVs** before saving if you want to replace the current detections.

In [10]:
if REDETECT_TRENCHES and store.has_trench_detection():
    store.reset("trench_detection")
if store.has_trench_detection():
    trench_det = TrenchDetector.load(exp, reg, lane_det, store)
    print(trench_det)
else:
    trench_det = TrenchDetector(
        exp, reg, lane_det, store, detection_channel=DETECTION_CHANNEL,
    )

TrenchDetector(7461 trenches across 75 FOVs)


In [11]:
trench_det.interactive_detect_trenches()

RFBOutputContext()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
if not trench_det.is_detected:
    raise RuntimeError("Click 'Apply to all FOVs' in the trench widget first.")
trench_det.save()
print(trench_det)

Trench detection saved to /data/20260331_SB5_6_7_8_16/20260331_restart.pymmm.zarr
TrenchDetector(7519 trenches across 75 FOVs)


## Extract

The first cell lazily previews the exact all-channel trench layout without writing it. Run the following cell to commit the extraction. Interrupted runs resume at the last completed FOV when `RESUME_EXTRACTION=True`.

In [14]:
extractor = Extractor(exp, reg, trench_det, output_path=OUTPUT_PATH)
print(extractor)
extraction_preview = extractor.preview()
extraction_ndv_viewer = ndv.imshow(_LocalXarrayWrapper(extraction_preview))
extraction_ndv_viewer

Extractor: 20260331_restart
  Output: /data/20260331_restart.fast.trenches.zarr
  Shape: Trench=7519 × T=97 × C=4 × Y=160 × X=32
  Chunks: (1, 97, 1, 160, 32)
  Dtype: uint16
  Size (uncompressed): 27.8 GB
  Trenches: 7519 across 75 FOVs
  Trench size: 160 × 32 px (17.3 × 3.5 µm)
  Channels: PC, CFP, mVenus, mCherry
  Timepoints: 97


RFBOutputContext()

<IPython.core.display.Javascript object>

In [15]:
extractor.extract(
    clevel=COMPRESSION_LEVEL,
    byte_shuffle=BYTE_SHUFFLE,
    batch_size=EXTRACTION_BATCH_SIZE,
    time_chunk=OUTPUT_TIME_CHUNK,
    resume=RESUME_EXTRACTION,
)

Extracting FOVs:   0%|          | 0/75 [00:00<?, ?it/s]

Extraction complete: /data/20260331_restart.fast.trenches.zarr
  Shape: (7519, 97, 4, 160, 32), Chunks: (1, 97, 1, 160, 32)


In [ ]:
client.close()
cluster.close()